# Survival Analysis — Time to Default by Loan Grade

**Question:** *Which grades default earliest, and by how much?*

Kaplan-Meier curves segmented by grade A→G, plus a Cox Proportional Hazards model so we can quantify hazard ratios for each covariate. Output: median survival time per grade + the gap between the safest and riskiest grade.

Run `python -m src.train` first if `outputs/model.pkl` doesn't exist.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import pandas as pd

from src import data_loader
from src import survival

df = data_loader.load_clean()
print(f'{len(df):,} loans loaded.')
df[['grade_letter','duration_months','event']].head()

## Kaplan-Meier curves by grade

In [ ]:
fitters, summary = survival.km_by_grade(df)

fig, ax = plt.subplots(figsize=(10, 6))
for grade, kmf in fitters.items():
    kmf.plot_survival_function(ax=ax, ci_show=False)
ax.set_xlabel('Months since loan issuance')
ax.set_ylabel('P(no default by month t)')
ax.set_title('Kaplan-Meier survival by grade', fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/km_by_grade.png', dpi=140, bbox_inches='tight')
plt.show()
summary

## Median survival gap — safest vs riskiest grade

In [ ]:
gap = survival.median_survival_gap(summary)
print(gap)

## Logrank tests — are the curves actually different?

In [ ]:
survival.pairwise_logrank(df)

## Cox Proportional Hazards

In [ ]:
covariates = ['fico', 'dti', 'int_rate', 'loan_to_income',
              'revol_util', 'grade_num', 'emp_length_num', 'credit_age_yrs']
cph = survival.fit_cox(df.sample(min(50000, len(df)), random_state=42), covariates)
cph.print_summary()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.set_title('Cox PH — log hazard ratios (95% CI)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/cox_hazard_ratios.png', dpi=140, bbox_inches='tight')
plt.show()